# S10 · Рэп, рок или поп: деревья и лес

6 ноября 2026. Выполняйте ячейки сверху вниз с чистого ядра. В каждом разделе напишите целую ячейку по условию.

## C01 · Читаем песни

Прочитайте data/russian-genres.csv. Одна строка это одна песня российского исполнителя: жанр и числа Spotify о звуке. Жанр записан у исполнителя, поэтому у всех его песен он один. [Данные и отбор](../data/GENRES-README.md).

In [1]:
# S10-C01
import csv
from pathlib import Path

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text

folder = Path.cwd()
if folder.name == "solutions":
    folder = folder.parent
if folder.name == "notebooks":
    folder = folder.parent
if folder.name != "starter":
    folder = folder / "starter"
with (folder / "data/russian-genres.csv").open(encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
print("Песен:", len(rows))
print(next(row for row in rows if row["track"] == "Что такое осень"))

Песен: 6034
{'track': 'Что такое осень', 'artist': 'ДДТ', 'year': '1992', 'genre': 'рок', 'split': 'test', 'popularity': '53', 'danceability': '0.602', 'energy': '0.799', 'loudness': '-12.171', 'speechiness': '0.0366', 'acousticness': '0.00307', 'instrumentalness': '0.000668', 'valence': '0.791', 'tempo': '94.218', 'minutes': '4.96'}


## C02 · Числа и части

Соберите таблицу X из 9 чисел и столбец жанров. Части заданы исполнителями: песни одного исполнителя лежат только в одной части, иначе модель узнавала бы голос и стиль знакомого артиста. Сравните с ленивым правилом «всегда рок».

In [3]:
# S10-C02
names = ["danceability", "energy", "loudness", "speechiness", "acousticness",
         "instrumentalness", "valence", "tempo", "minutes"]

X = np.array([[float(row[name]) for name in names] for row in rows])
y = np.array([row["genre"] for row in rows])
part = np.array([row["split"] for row in rows])
train, validation, test, usa = [part == name for name in ["train", "validation", "test", "usa"]]
print("train, validation, test, usa:", train.sum(), validation.sum(), test.sum(), usa.sum())
print("Всегда рок на validation:", round((y[validation] == "рок").mean(), 3))

train, validation, test, usa: 2627 1526 1595 286
Всегда рок на validation: 0.531


## C03 · Дерево алгоритма

На слайде мы выбирали первый вопрос из четырёх. Алгоритм делает то же самое, но перебирает все числа и пороги и берёт вопрос, после которого кучки чище. Глубина 3, то есть не больше трёх вопросов подряд, выбрана заранее. Напечатайте вопросы и точность.

In [4]:
# S10-C03
tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X[train], y[train])
print(export_text(tree, feature_names=names))
print("train:", round(tree.score(X[train], y[train]), 3))
print("validation:", round(tree.score(X[validation], y[validation]), 3))

|--- speechiness <= 0.12
|   |--- danceability <= 0.65
|   |   |--- loudness <= -9.73
|   |   |   |--- class: рок
|   |   |--- loudness >  -9.73
|   |   |   |--- class: рок
|   |--- danceability >  0.65
|   |   |--- valence <= 0.57
|   |   |   |--- class: рэп
|   |   |--- valence >  0.57
|   |   |   |--- class: рок
|--- speechiness >  0.12
|   |--- energy <= 0.87
|   |   |--- loudness <= -9.98
|   |   |   |--- class: рэп
|   |   |--- loudness >  -9.98
|   |   |   |--- class: рэп
|   |--- energy >  0.87
|   |   |--- danceability <= 0.53
|   |   |   |--- class: рок
|   |   |--- danceability >  0.53
|   |   |   |--- class: рэп

train: 0.671
validation: 0.712


## C04 · Дерево наизусть

Уберите ограничение: дерево спрашивает, пока в каждой кучке не останется один жанр. Сравните train и validation.

In [5]:
# S10-C04
free_tree = DecisionTreeClassifier(random_state=0)
free_tree.fit(X[train], y[train])
print("Листьев:", free_tree.get_n_leaves())
print("train:", round(free_tree.score(X[train], y[train]), 3))
print("validation:", round(free_tree.score(X[validation], y[validation]), 3))

Листьев: 523
train: 1.0
validation: 0.647


## C05 · Две половины

Обучите два таких дерева на случайных половинах train. На какой доле песен validation они дают разный жанр? Это дисперсия из L4.

In [6]:
# S10-C05
rng = np.random.default_rng(10)
ids = rng.permutation(np.where(train)[0])
half = len(ids) // 2
first = DecisionTreeClassifier(random_state=0).fit(X[ids[:half]], y[ids[:half]])
second = DecisionTreeClassifier(random_state=0).fit(X[ids[half:]], y[ids[half:]])
print("Спорят на доле песен:", round((first.predict(X[validation]) != second.predict(X[validation])).mean(), 3))

Спорят на доле песен: 0.36


## C06 · Лес и экзамен

Случайный лес: много деревьев, каждое на своей случайной выборке песен с повторами, ответ выбирается голосованием. 300 деревьев выбраны заранее. Сравните на validation, затем один раз откройте test.

In [7]:
# S10-C06
forest = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)
forest.fit(X[train], y[train])
models = {"дерево из 3 вопросов": tree, "дерево наизусть": free_tree, "лес": forest}
for name, model in models.items():
    print(name, round(model.score(X[validation], y[validation]), 3), round(model.score(X[test], y[test]), 3))
print("всегда рок", round((y[validation] == "рок").mean(), 3), round((y[test] == "рок").mean(), 3))

дерево из 3 вопросов 0.712 0.642
дерево наизусть 0.647 0.612
лес 0.769 0.72
всегда рок 0.531 0.468


## C07 · Что сказал лес

Посмотрите голоса леса за жанры на песнях из начала пары. Все эти исполнители лежат в test, лес их не видел.

In [8]:
# S10-C07
songs = ["Life", "чайлдфри", "Город под подошвой", "Что такое осень", "KARI", "молчи"]
for track in songs:
    index = next(i for i, row in enumerate(rows) if row["track"] == track and row["split"] == "test")
    votes = forest.predict_proba(X[[index]])[0]
    print(track, rows[index]["genre"], dict(zip(forest.classes_.tolist(), votes.round(2).tolist())))

Life поп {'поп': 0.07, 'рок': 0.01, 'рэп': 0.92}
чайлдфри рэп {'поп': 0.62, 'рок': 0.36, 'рэп': 0.02}
Город под подошвой рэп {'поп': 0.03, 'рок': 0.61, 'рэп': 0.36}
Что такое осень рок {'поп': 0.11, 'рок': 0.89, 'рэп': 0.0}
KARI рэп {'поп': 0.21, 'рок': 0.03, 'рэп': 0.76}
молчи рок {'поп': 0.17, 'рок': 0.25, 'рэп': 0.57}


## C08 · Американский рэп

Лес учился только на российских песнях. Дайте ему 286 популярных песен американских рэперов. Какую долю он назовёт рэпом?

In [9]:
# S10-C08
answers = forest.predict(X[usa])
print("Назвал рэпом:", round((answers == "рэп").mean(), 3))
print({genre: int((answers == genre).sum()) for genre in forest.classes_.tolist()})

Назвал рэпом: 0.79
{'поп': 26, 'рок': 34, 'рэп': 226}
